# Temperature Prediction - Machine Learning Project
## S403011 Machine Learning - University of Geneva

This notebook predicts air temperature at three horizons (12h, 24h, 48h) using meteorological data from Swiss weather stations.

---
## 0) Imports and Setup

In [ ]:
import pandas as pd  # data loading, manipulation and analysis
import numpy as np  # numerical computations, vector operations

import matplotlib.pyplot as plt  # creation of plots and statistical visualizations
import seaborn as sns  # advanced statistical visualizations (e.g., correlation matrices)

from sklearn.preprocessing import RobustScaler  # robust features scaling

from sklearn.linear_model import LinearRegression, Ridge  # linear and ridge regression fitting
from sklearn.ensemble import RandomForestRegressor, \
    GradientBoostingRegressor  # random forest and gradient boosting fitting
from sklearn.base import clone # duplicate model fitting

from sklearn.metrics import mean_absolute_error  # for the post-model computation of MAE

from sklearn.model_selection import (
    train_test_split,  # dataset splitting
    cross_validate,  # model evaluation via cross-validation
    GridSearchCV,  # hyperparameter tuning
    KFold  # standard cross-validation
)

from sklearn.compose import ColumnTransformer  # applies feature-specific preprocessing steps
from sklearn.pipeline import Pipeline  # pipelines manipulation

from sklearn.impute import SimpleImputer  # train-only imputation (leakage-safe)
from sklearn.preprocessing import (
    FunctionTransformer,  # feature transformations (leakage-safe when pipelined)
    OneHotEncoder  # leakage-safe categorical encoding
)

# Display options (set once at the beginning) :
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

---
## 1) EDA-Relevant Preprocessing

### 1.1) Reorganize our dataframe

In [ ]:
print("\n" + "=" * 60)
print("EDA-relevant PREPROCESSING")
print("=" * 60)

dataset = pd.read_csv("train.csv")

The following function will allow us to get knowledge about the number of both variables and observations, the columns' names, their types, and eventual missing values for any of them.

In [ ]:
print("Here is the original dataset information:\n")
dataset.info()

To make visualization and data analysis more readable, we resort the columns so that all variables are grouped by their meteorological station.

In [ ]:
stations = ["ANT", "BAS", "DAV", "DOL", "GVE", "INT", "LUG", "SIO", "STG", "ZER"]
dataset = dataset[sorted(dataset.columns, key=lambda c: c.split("_")[-1] if "_" in c else "ZZZ")]

We also rename the variables to reflect their meaning (as specified in the competition's "data" section on Kaggle).

In [ ]:
mapping = {
    "tre200h0": "air_temp_2m",
    "tre200h0_lag24h": "air_temp_2m_lag24h",
    "ure200h0": "rel_humidity_2m",
    "fkl010h0": "wind_speed",
    "fkl010h3": "wind_gust",
    "gre000h0": "global_radiation",
    "pp0qffh0": "pressure_sea_level",
    "prestah0": "pressure_barometric",
    "rre150h0": "precipitation",
    "sre000h0": "sunshine_duration",
    "target_tre200h0_plus12h": "target_air_temp_2m_plus12h",
    "target_tre200h0_plus24h": "target_air_temp_2m_plus24h",
    "target_tre200h0_plus48h": "target_air_temp_2m_plus48h"
}
dataset.rename(columns=lambda col:
next((col.replace(old, new) for old, new in mapping.items() if col.startswith(old)), col)
               , inplace=True)

Let's now re-execute our previous function, as the dataset became now better specified and organized:

In [ ]:
print("\nLet's now display the reorganized dataset information:\n")
dataset.info()

We will create a raw dataset, free of the processing we will perform in the following sections (in particular, the imputation of NaN).

Later, when building our models, any model-relevant preprocessing step must only be applied to the training data in order to prevent data-leakage.

The same transformations will then be applied to the test data, by using the parameters learned from the training set.

In [ ]:
raw_dataset = dataset.copy()

### 1.2) Dealing with missing values

As we can see from the "Non-Null Count" of `dataset.info()`, almost every variable has between 1 and 4 missing values.

We need to adequately impute or remove these values.

**Let's start with the targets.**

Training on features with missing target values would distort the model. Therefore, we will not attempt to impute NaN in the targets, and will simply remove the rows where targets are missing.

In [ ]:
dataset = dataset.dropna(subset=[
    'target_air_temp_2m_plus12h',
    'target_air_temp_2m_plus24h',
    'target_air_temp_2m_plus48h'
])

We now have a new dataset with slightly few observations, but still many missing values in the features.

**Regarding the features**, using `dropna` is usually not the best option.

Whenever one or more features are missing for a given observation, using `dropna` removes all the values of the features that are actually available. For this reason, we only apply `dropna` to rows with many missing values. Otherwise, we prefer imputation strategies: by replacing the missing value with a plausible estimate, we can preserve the remaining feature values for that observation.

To assess whether we should apply imputation techniques rather than remove rows containing missing values, let's display the number of missing values in each row that contains at least one.

In [ ]:
nan_per_row = dataset.isna().sum(axis=1)
print("\nHere below, we have the number of missing values for each observation that contains at least one:\n")
print(nan_per_row[nan_per_row > 0])

From this output, we see that none of the rows contains more than one missing value. Thus, `dropna` is not an optimal solution, and we should instead use imputation methods.

As we have few missing values for each feature (never more than 4 out of 7,575 observations), we can simply estimate them using a measure of central tendency for the column. This method will have little effect on its distribution.

For each of these variables, taking a value that is representative of the centre of its distribution is an optimal choice, as it minimises the error in this estimate.

Let's first handle the missing values for our only categorical variable, `season`. We will impute its single missing value by the feature's mode.

In [ ]:
dataset.loc[:, 'season'] = dataset['season'].fillna(dataset['season'].mode()[0])

For quantitative variables, estimating missing values using the mean of the distribution is an optimal choice. The mean incorporates information from the entire distribution, not just the order of values.

If there are too many outliers, however, the distribution becomes skewed, distorting the mean as an estimate of the population expectation. In this case, we prefer to use the median.

We use **Tukey's rule** to assess whether a given observations should be considered as an outlier: if some values of feature outside the upper/lower whisker, they are flagged as outliers.

If we have more than 1% of outliers for some variable, we will apply the median to impute the NaN on this feature.

In [ ]:
quant_features = dataset.select_dtypes(include=[np.number]).columns.tolist()
targets = [
    "target_air_temp_2m_plus12h",
    "target_air_temp_2m_plus24h",
    "target_air_temp_2m_plus48h"
]
# Remove targets from the 'quant_features' variable:
quant_features = [col for col in quant_features if col not in targets]

In [ ]:
def impute_numeric_features(df, quant_features, outlier_threshold=0.01):
    """Impute missing values using mean or median based on outlier proportion."""
    X = df.copy()
    many_outliers = []
    few_outliers = []
    for col in quant_features:
        Q1 = X[col].quantile(0.25)
        Q3 = X[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_whisker = Q1 - 1.5 * IQR
        upper_whisker = Q3 + 1.5 * IQR
        outliers_mask = (X[col] < lower_whisker) | (X[col] > upper_whisker)
        outlier_ratio = outliers_mask.mean()
        # We just built the Tukey's criteria for what should be considered as outlier.
        if outlier_ratio > outlier_threshold:
            value = X[col].median()
            many_outliers.append(col)
        else:
            value = X[col].mean()
            few_outliers.append(col)
        X[col] = X[col].fillna(value)
    return X, many_outliers, few_outliers

In [ ]:
dataset, many_outliers, few_outliers = impute_numeric_features(dataset, quant_features)

print("\nVariables with many outliers:")
print(many_outliers)

print("\nVariables with few outliers:\n")
print(few_outliers)

print("\nHere is the reorganized dataset information with proper imputation of missing values:\n")
dataset.info()

### 1.3) Handling outliers

We must evaluate whether outliers are problematic.

Looking at the features with many outliers, we notice that they correspond to meteorological variables prone to seasonal peaks. This could explain why their distributions contain many extreme values. Let's check to be sure.

In [ ]:
print("\nMin/max for variables with many outliers:\n")
for col in many_outliers:
    col_min = dataset[col].min()
    col_max = dataset[col].max()
    print(f"{col}: min = {col_min:.2f}, max = {col_max:.2f}")

We can confirm that these ranges fall within realistic physical limits for these variables. Therefore, these outliers are likely not due to measurement errors and instead represent meaningful meteorological extremes. Consequently, we choose not to remove them.

**Note:** we only examined the ranges of the features most sensitive to outliers. For the remaining variables, we didn't perform additional checks (their outlier proportions are low and therefore do not pose a significant issue).

To conclude, our outliers represent relevant predictors and are relevant for the future EDA. 

### 1.4) Handling temporal variables

In [ ]:
print("\nLet's now look at the order of temporal variables:\n")
print(dataset[["hour", "season"]].head(50))

It quickly becomes apparent that the seasons and times are not chronologically consistent. Without third-party variables, such as the date, it is impossible for us to restore the chronological order of the observations.

**For this reason, we cannot treat our weather dataset as a time series.**

However, minimal elaboration of our temporal variables is required.

In [ ]:
def create_season_dummies(df, column="season", drop_first=True, prefix="season"):
    df = df.copy()
    dummies = pd.get_dummies(df[column], drop_first=drop_first, prefix=prefix)
    df = pd.concat([df.drop(columns=[column]), dummies], axis=1)
    return df

dataset = create_season_dummies(dataset, column="season", drop_first=True)

We converted the variable `season` into 3 binary dummies (one is left as an implicit reference, to avoid multicollinearity).

**Note:** we will not use these new features in the future scaling, as these are dummies (the same will apply for `hour_sin`/`hour_cos`).

We also transform the variable `hour` into two cyclical features to better capture the daily cycle.

In [ ]:
def add_hour_cyclical_features(df, hour_column="hour"):
    df = df.copy()
    hours = df[hour_column].astype(float)
    df[f"{hour_column}_sin"] = np.sin(2 * np.pi * hours / 24.0)
    df[f"{hour_column}_cos"] = np.cos(2 * np.pi * hours / 24.0)
    df = df.drop(columns=[hour_column])
    return df

dataset = add_hour_cyclical_features(dataset, hour_column="hour")

In [ ]:
print("\nDataset information with adjusted temporal features (EDA ready):\n")
dataset.info()

Let's build a variable with all the predictors:

In [ ]:
features = [c for c in dataset.columns if c not in targets]

---
## 2) Exploratory Data Analysis

In this section, we will investigate the entire, non-standardized weather dataset. This will allow us to gather as many insights as possible about the structure of our weather data.

The Pearson correlation coefficients, that we will use, are robust to scale.

Since Pearson's correlation measures linear relationships, the effects of cyclical variables such as `hour_sin` and `hour_cos` should be interpreted with care.

In [ ]:
print("\n" + "=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

### 2.1) Correlation among features

We are willing to detect multicollinearity among features.

In [ ]:
# Pearson correlation matrix between all features:
corr_matrix = dataset[features].corr()

# Because it is symmetric, we extract the upper triangle of the correlation matrix (excluding the diagonal) to avoid redundancy:
upper_triangle = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]

print("\nSome statistics on pairwise correlations among features:")
print(f"Most negative correlation: {upper_triangle.min():.3f}")
print(f"Most positive correlation: {upper_triangle.max():.3f}")
print(f"Lowest correlation in absolute value: {np.abs(upper_triangle).min():.3f}")
print(f"Mean correlation: {upper_triangle.mean():.3f}")
print(f"Std correlation: {upper_triangle.std():.3f}\n")

print("Now, let's look for potential multicollinearity.\n")

We display highly correlated feature pairs:

In [ ]:
high_corr_pairs = []
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.85:
            f1 = corr_matrix.columns[i]
            f2 = corr_matrix.columns[j]
            high_corr_pairs.append((f1, f2, val))

if high_corr_pairs:
    print("\n ⚠️ Highly correlated feature pairs (|r| > 0.85):")
    for f1, f2, corr_val in high_corr_pairs[:15]:  # Show first 15
        print(f" {f1} <-> {f2}: {corr_val:.3f}")

As we can see, several pairs of features are affected by multicollinearity.

To make this information easier to read, we will construct correlation matrices corresponding to this data. For predefined feature groups, the following function generates correlation matrices, and selectively plots only those containing at least one correlation above 0.85. In these visualizations, it displays exclusively the correlation values exceeding this threshold.

In [ ]:
def plot_filtered_correlation_matrices_vector(
        dataset, list_of_variable_groups, threshold=0.85,
        output_file="correlation_blocks.pdf", title_prefix="",
        title_suffix="", show_titles=False):
    corr_matrices = []
    titles = []
    for group in list_of_variable_groups:
        if len(group) < 2:
            continue
        corr_matrix = dataset[group].corr()
        upper = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)]
        if not (np.abs(upper) >= threshold).any():
            continue
        corr_matrices.append(corr_matrix.mask(np.abs(corr_matrix) < threshold))
        titles.append(title_prefix + group[0].rsplit("_", 1)[0].replace("_", " ").title() + title_suffix)
    if len(corr_matrices) == 0:
        return
    n = len(corr_matrices)
    cols_grid = 2
    rows_grid = -(-n // cols_grid)
    fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(8 * cols_grid, 6 * rows_grid))
    axes = np.atleast_1d(axes).flatten()
    for i, (corr_mat, title) in enumerate(zip(corr_matrices, titles)):
        ax = axes[i]
        sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm",
                    vmin=-1, vmax=1, linewidths=.5, linecolor='gray', cbar=False, ax=ax)
        if show_titles:
            ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xticklabels(corr_mat.columns, rotation=45, ha='right', fontsize=10)
        ax.set_yticklabels(corr_mat.index, fontsize=10)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    plt.tight_layout()
    plt.savefig(output_file, format='pdf', bbox_inches="tight", dpi=300)
    plt.close()

#### 2.1.1) Geographical multicollinearity

As indicated by the previous print(), several features exhibit high correlations across stations.

To quantify inter-station collinearity, we compute correlation matrices for the key meteorological variables. We specifically focus on the features affected by geographical multicollinearity, in order to limit the number of plots in the report.

In [ ]:
sns.set(style="white")
key_variables = [
    "air_temp_2m",
    "rel_humidity_2m",
    "wind_speed",
    "wind_gust",
    "global_radiation",
    "pressure_sea_level",
    "pressure_barometric",
    "precipitation",
    "sunshine_duration"
]

geographical_groups = []
for var in key_variables:
    cols = [c for c in dataset.columns if c.startswith(var + "_")]
    if len(cols) >= 2:
        geographical_groups.append(cols)

output_geo = "geographical_correlations.pdf"
plot_filtered_correlation_matrices_vector(
    dataset=dataset,
    list_of_variable_groups=geographical_groups,
    threshold=0.85,
    output_file="geographical_correlations.pdf",
    title_prefix="Inter-Station Corr (|r| > 0.85): ",
    show_titles=True
)
print(f"\nThe inter-station correlation matrices have been saved under: {output_geo}\n")

#### 2.1.2) Structural multicollinearity

Now, we want to build a structural, non-geographical correlation matrix. This time, we focus on:
- the correlation among different features, and
- the correlation between a station-specific feature and its corresponding value in Bern (in our case, we are speaking about `air_temp_2m`).

In [ ]:
geo_pairs = set()
for group in geographical_groups:
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            geo_pairs.add(tuple(sorted([group[i], group[j]])))

all_cols = list(dataset.columns)
remaining_groups = []
current_group = []
for col in all_cols:
    can_add = True
    for existing in current_group:
        pair = tuple(sorted([col, existing]))
        if pair in geo_pairs:
            can_add = False
            break
    if can_add:
        current_group.append(col)
    else:
        remaining_groups.append(current_group)
        current_group = [col]
if current_group:
    remaining_groups.append(current_group)

final_groups = []
for group in remaining_groups:
    for i in range(0, len(group), 20):
        final_groups.append(group[i:i + 20])

output_struct = "structural_correlations.pdf"
plot_filtered_correlation_matrices_vector(
    dataset=dataset,
    list_of_variable_groups=final_groups,
    threshold=0.85,
    output_file="structural_correlations.pdf",
    title_prefix="",
    title_suffix="",
    show_titles=False
)
print(f"\nThe structural correlation matrix (in blocks) has been saved under: {output_struct}\n")

Due to the large number of variables included in each station-level correlation matrix, an overall title is omitted to maintain readability.

### 2.2) Correlation analysis between features and targets

In [ ]:
print("\nLet's now focus on the correlation between each feature and the targets.")
print(f"Total remaining features: {len(features)}")
print(f"Total targets: {len(targets)}\n")

Let's find for which features we have a significant correlation with each of the targets:

In [ ]:
threshold = 0.3  # Pearson correlation threshold
for target in targets:
    print(f"Features with absolute correlation above {threshold} for: {target}")
    correlations = dataset[features].corrwith(dataset[target])
    filtered_corr = correlations[correlations.abs() >= threshold].sort_values(ascending=False)
    for i, (name, value) in enumerate(filtered_corr.items(), 1):
        print(f" {i:2d}. {name:45s} : {value:7.4f}")
    print()

Now, let's plot these most significant correlations.

For the 12-hour target, we keep the threshold at 0.3 to choose which features to plot. For 24 and 48 hours, we use a threshold of 0.4 to limit the complexity of the plots (otherwise, we would consider too many predictors).

In [ ]:
thresholds = {
    'target_air_temp_2m_plus12h': 0.3,
    'target_air_temp_2m_plus24h': 0.4,
    'target_air_temp_2m_plus48h': 0.4
}

main_features_12h = []
main_features_24h = []
main_features_48h = []

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for idx, target in enumerate(targets):
    threshold = thresholds[target]
    correlations = dataset[features].corrwith(dataset[target])
    filtered_corr = correlations[correlations.abs() >= threshold].sort_values(ascending=False)

    if target == 'target_air_temp_2m_plus12h':
        main_features_12h = filtered_corr.index.tolist()
    elif target == 'target_air_temp_2m_plus24h':
        main_features_24h = filtered_corr.index.tolist()
    elif target == 'target_air_temp_2m_plus48h':
        main_features_48h = filtered_corr.index.tolist()

    colors = ['green' if x > 0 else 'red' for x in filtered_corr.values]

    axes[idx].barh(range(len(filtered_corr)), filtered_corr.values, color=colors, alpha=0.7)
    axes[idx].set_yticks(range(len(filtered_corr)))
    axes[idx].set_yticklabels(filtered_corr.index, fontsize=9)
    axes[idx].set_xlabel('Correlation', fontsize=11)
    axes[idx].set_title(target.replace('target_', '').replace('_', ' ').title(),
                        fontsize=12, fontweight='bold')
    axes[idx].axvline(x=0, color='black', linewidth=1.5)
    axes[idx].grid(axis='x', alpha=0.3)

plt.suptitle('Feature Correlations with the Targets Variables (for the most significant correlations)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
output_file = 'target_correlations_thresholded.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.close()
print(f"\nThe barplot for significant features has been saved under: {output_file}\n")

Let's look now, for each key-variable, at the correlation between targets and the different stations.

In [ ]:
n_vars = len(key_variables)
nrows = (n_vars + 1) // 2
ncols = 2
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 3 * nrows))
axes = axes.flatten()
for i, var in enumerate(key_variables):
    if i >= n_vars:
        break
    corr_dataset = pd.DataFrame(index=stations, columns=targets, dtype=float)
    for station in stations:
        col_name = f"{var}_{station}"
        if col_name not in dataset.columns:
            continue
        for target in targets:
            if target not in dataset.columns:
                continue
            corr_dataset.loc[station, target] = dataset[col_name].corr(dataset[target])
    station_order = corr_dataset.abs().mean(axis=1).sort_values(ascending=False).index
    corr_dataset = corr_dataset.loc[station_order]
    sns.heatmap(corr_dataset, annot=True, cmap="RdBu_r", center=0, ax=axes[i],
                fmt='.2f', cbar_kws={'shrink': 0.6},
                annot_kws={'size': 8})
    axes[i].set_title(f"{var}", fontsize=10, pad=5)
    axes[i].tick_params(axis='both', labelsize=8)
for i in range(n_vars, len(axes)):
    axes[i].set_visible(False)
plt.tight_layout(pad=1.0)
output_target = "targ_stat_corr.pdf"
plt.savefig(output_target, dpi=300, bbox_inches='tight')
plt.close()
print(f"\nThe heatmap for targets vs stations correlations has been saved under: {output_target}\n")

A proper interpretation of the previous plots will be done in the report.

The following lines aim to create a new variable, containing only the most relevant features.

The `reduced_feature` variable will replace the variable `features`, in the rest of the EDA, in order to have a better comprehension of the main explanatory variables.

**How will we build the variable `reduced_features`?**

For any high correlated pair of variable (|r| > 0.85), we will remove from `features` the variable which correlation mean with the targets is the lowest (to avoid multicollinearity). However, this doesn't apply if the high correlation involves either 2 seasons, or `hour_sin` and `hour_cos`.

In [ ]:
def reduce_multicollinearity(dataset, features, targets, threshold):
    feature_target_corrs = (
        dataset[features + targets]
        .corr()
        .loc[features, targets]
        .abs()
        .mean(axis=1)
        .to_dict()
    )
    corr_matrix = dataset[features].corr().abs()
    to_remove = set()
    for i in range(len(corr_matrix)):
        for j in range(i + 1, len(corr_matrix)):
            if corr_matrix.iloc[i, j] >= threshold:
                f1 = corr_matrix.columns[i]
                f2 = corr_matrix.columns[j]
                both_season = f1.startswith("season_") and f2.startswith("season_")
                both_hour = set([f1, f2]) == set(["hour_sin", "hour_cos"])
                if both_season or both_hour:
                    continue

                if f1 in to_remove or f2 in to_remove:
                    continue
                if feature_target_corrs[f1] >= feature_target_corrs[f2]:
                    to_remove.add(f2)
                else:
                    to_remove.add(f1)
    reduced_features = [f for f in features if f not in to_remove]
    return reduced_features

reduced_features_eda = reduce_multicollinearity(dataset, features, targets, 0.85)
print("\nAfter resolving multicollinearity, here is the list of reduced features:", len(reduced_features_eda), "features.\n")

### 2.3) Distribution analysis for the most important features

Let's take the features for which |r|>0.4 with confront to the target at 24 hours (the target we focus on in the kaggle competition).

In [ ]:
target = 'target_air_temp_2m_plus24h'
threshold = 0.4
correlations = dataset[reduced_features_eda].corrwith(dataset[target])
selected_features = correlations[correlations.abs() >= threshold].sort_values(ascending=False).index.tolist()

We plot the statistical distributions for these features:

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for idx, var in enumerate(selected_features):
    if idx >= len(axes):
        break
    data = dataset[var].dropna()
    if data.dtype == 'bool':
        data = data.astype(int)
    axes[idx].hist(data, bins=50, alpha=0.6, color='skyblue', edgecolor='black', density=True)
    data.plot(kind='kde', ax=axes[idx], color='red', linewidth=2)
    mean_val = data.mean()
    median_val = data.median()
    skew_val = data.skew()
    std_val = data.std()
    axes[idx].axvline(mean_val, color='green', linestyle='--', linewidth=2)
    axes[idx].axvline(median_val, color='orange', linestyle='--', linewidth=2)
    axes[idx].set_title(f'{var}\nSkew: {skew_val:.2f} | Std: {std_val:.2f}', fontsize=10, fontweight='bold')
    axes[idx].tick_params(labelsize=8)
    axes[idx].grid(alpha=0.3)
for idx in range(len(selected_features), len(axes)):
    axes[idx].axis('off')
plt.suptitle(f'Distribution Analysis - Features Correlated ≥ {threshold} with Target 24h',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('distribution_analysis_24h_features.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"\nDistribution plots for the main features have been saved under: distribution_analysis_24h_features.png\n")

### 2.4) Target exploration

By looking at the distributions of the targets, we will be able to assess whether a regression on the features would appropriately predict them. Moreover, this helps identify potential skewness or bimodalities.

In [ ]:
fig, axes = plt.subplots(1, len(targets), figsize=(18, 5))
for idx, target in enumerate(targets):
    sns.kdeplot(dataset[target].dropna(), ax=axes[idx], fill=True, color='skyblue', linewidth=2)
    axes[idx].axvline(dataset[target].mean(), color='green', linestyle='--', linewidth=2)
    axes[idx].axvline(dataset[target].median(), color='orange', linestyle='--', linewidth=2)
    axes[idx].set_title(target.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Temperature (°C)', fontsize=9)
    axes[idx].set_ylabel('Density', fontsize=9)
    axes[idx].grid(alpha=0.3)
plt.suptitle('KDE of Target Temperatures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('targets_kde.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"\nDistribution plots for the targets have been saved under: targets_kde.png\n")

The following descriptive statistics will allow us to assess the variability of the target variables. 

In [ ]:
print("\nHere are the main descriptive statistics for the target variables:\n")
print(dataset[targets].describe().T)

A study of the correlations between the target horizons could have provided additional insights into how these variables evolve together over time. However, due to space constraints and because our project relies on simple, non-multi-horizon models, we decided to leave this analysis aside.

We also omitted to plot hourly and seasonal patterns, as it would have required even more visual space.

---
## 3) Train/test split and model-relevant preprocessing

We now prepare the data for statistical modeling. The same preprocessing steps as in the EDA phase are applied, with three key differences:

**a) Feature standardization:**
The training features have heterogeneous scales due to differences in measurement units and ranges. Such discrepancies may bias certain models, particularly distance-based and regularized methods. Therefore, all features are standardized to ensure a consistent scale.

**b) Prevention of data leakage:**
Any preprocessing step involving data transformation (imputation and standardization) is fitted exclusively on the training data and then applied to the test data. This prevents information leakage from the test set into the training process.

**c)** To make the imputation process fully compatible with the pipeline fitting, and considering the presence of outliers, numerical features were imputed using the median. This approach leverages scikit-learn's `SimpleImputer`, ensuring that imputation is fitted only on the training data, thus preventing data leakage. 

In [ ]:
raw_dataset = raw_dataset.dropna(subset=targets)
X = raw_dataset.drop(columns=targets)
y = raw_dataset[targets]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

**Relevant functions for the modelling process:**

In [ ]:
# One-hot encoding is applied within the scikit-learn pipeline to prevent data leakage.
def _make_onehot_encoder():
    try:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

In [ ]:
def hour_to_cyclical(X):
    arr = np.asarray(X).astype(float).reshape(-1)
    sin = np.sin(2 * np.pi * arr / 24.0)
    cos = np.cos(2 * np.pi * arr / 24.0)
    return np.column_stack([sin, cos])

In [ ]:
# Leakage-safe preprocessor:
# SimpleImputer is fitted only on the training data, while hourly and seasonal features are transformed as in the EDA-relevant preprocessing.
def make_preprocessor(features, numeric_cols, scale_numeric=True):
    special_cols = {"hour", "season"}
    numeric_cols_clean = [c for c in numeric_cols if c not in special_cols]
    transformers = []
    # Numeric features (excluding 'hour' and 'season')
    if numeric_cols_clean:
        num_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            num_steps.append(("scaler", RobustScaler()))
        transformers.append((
            "num",
            Pipeline(num_steps),
            numeric_cols_clean
        ))
    # Hour feature (cyclical)
    if "hour" in features:
        transformers.append((
            "hour",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("hour_cyc", FunctionTransformer(hour_to_cyclical, validate=False))
            ]),
            ["hour"]
        ))
    # Season feature (categorical)
    if "season" in features:
        transformers.append((
            "season",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", _make_onehot_encoder())
            ]),
            ["season"]
        ))
    return ColumnTransformer(transformers=transformers, remainder="drop")

In [ ]:
# Model evaluation pipeline with training and evaluation CV MAE, final train MAE and predictions on test set.
def evaluate_model(pipeline, X_tr, y_tr, X_te, cv):
    cv_results = cross_validate(
        pipeline, X_tr, y_tr,
        scoring="neg_mean_absolute_error",
        cv=cv,
        return_train_score=True,
        n_jobs=1
    )
    mae_train_cv = -cv_results["train_score"].mean()
    mae_val_cv = -cv_results["test_score"].mean()
    pipeline.fit(X_tr, y_tr)
    y_pred_tr = pipeline.predict(X_tr)
    mae_train_final = mean_absolute_error(y_tr, y_pred_tr)
    y_pred_te = pipeline.predict(X_te)
    return pipeline, mae_train_cv, mae_val_cv, mae_train_final, y_pred_te

In [ ]:
# Recovering features names from ColumnTransformer, useful for coefficient plots.
def get_feature_names_from_preprocessor(preprocessor, X):
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if transformer == "drop" or transformer is None:
            continue

        if name == "num":
            feature_names.extend(list(cols))

        elif name == "hour":
            base = cols[0]
            feature_names.extend([f"{base}_sin", f"{base}_cos"])

        elif name == "season":
            try:
                ohe = transformer.named_steps["onehot"]
                feature_names.extend(list(ohe.get_feature_names_out(cols)))
            except Exception:
                dummies = pd.get_dummies(X[cols[0]], drop_first=True, prefix="season")
                feature_names.extend(list(dummies.columns))
    return feature_names

In [ ]:
# Feature importance visualization (model-dependent coefficients or importances).
def plot_top_features(ax, feature_names, values, title, top_k=10):
    coef_series = pd.Series(values, index=feature_names)
    top_coef = coef_series.reindex(
        coef_series.abs().sort_values(ascending=False).index
    ).head(top_k)
    ax.bar(top_coef.index, top_coef.values, alpha=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(len(top_coef)))
    ax.set_xticklabels(top_coef.index, rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

---
## 4) Feature engineering

The following feature engineering steps will not be applied to all models, primarily for computational efficiency.

Given the high degree of both geographical and structural multicollinearity, adding interactions or anomaly-based features could negatively affect linear models. The transformations we apply are therefore better suited for non-parametric models, which can exploit these complex features without introducing bias or instability.

In [ ]:
print("\n" + "=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

### 4.1) Station aggregation

Given the strong inter-station correlations observed in the EDA, aggregating station-specific variables introduces minimal bias while substantially reducing predictor dimensionality and variance, leading to more stable model comprehensive and unbiased predictions.

In [ ]:
# Detection of station-specific features.
def _station_suffix(col: str):
    if "_" not in col:
        return None
    suf = col.rsplit("_", 1)[-1]
    return suf if suf in stations else None

# For these, removal of the station-characterizing suffix.
def _base_var(col: str):
    suf = _station_suffix(col)
    if suf is None:
        return None
    return col.rsplit("_", 1)[0]

In [ ]:
# Creation of inter-station aggregates (mean, std, min, max, range) for each station-specific key-variable.
# These variables will give us the meteorological general trends, variability and local extremes across stations.
def add_station_aggregates(X_: pd.DataFrame) -> pd.DataFrame:
    Xf = X_.copy()
    station_cols = [c for c in Xf.columns if _station_suffix(c)]
    if not station_cols:
        return Xf
    base_vars = sorted({_base_var(c) for c in station_cols if _base_var(c) is not None})
    for base in base_vars:
        cols = [f"{base}_{st}" for st in stations if f"{base}_{st}" in Xf.columns]
        if len(cols) >= 2:
            vals = Xf[cols]
            Xf[f"{base}__mean_stations"] = vals.mean(axis=1)
            Xf[f"{base}__std_stations"] = vals.std(axis=1)
            Xf[f"{base}__min_stations"] = vals.min(axis=1)
            Xf[f"{base}__max_stations"] = vals.max(axis=1)
            Xf[f"{base}__range_stations"] = Xf[f"{base}__max_stations"] - Xf[f"{base}__min_stations"]
    return Xf

### 4.2) Temperature dynamics

As confirmed by the EDA, `air_temp_2m` and `air_temp_2m_lag24h` are important predictors.

For each station, as well as for Bern, we create a 24-hour temperature variation feature: ΔT = T_now - T_lag24h, which captures short-term thermal dynamics.

In [ ]:
def add_temp_delta_24h(X_: pd.DataFrame) -> pd.DataFrame:
    Xf = X_.copy()
    # Compute delta per station:
    for st in stations:
        t_now = f"air_temp_2m_{st}"
        t_lag = f"air_temp_2m_lag24h_{st}"
        if t_now in Xf.columns and t_lag in Xf.columns:
            Xf[f"air_temp_2m__delta24h_{st}"] = Xf[t_now] - Xf[t_lag]
    # Compute delta for Bern column:
    if "air_temp_2m" in Xf.columns and "air_temp_2m_lag24h" in Xf.columns:
        Xf["air_temp_2m__delta24h"] = Xf["air_temp_2m"] - Xf["air_temp_2m_lag24h"]
    return Xf

### 4.3) Applying leakage-safe feature engineering

In [ ]:
def add_fe_features_basic(X_: pd.DataFrame) -> pd.DataFrame:
    Xf = X_.copy()
    Xf = add_station_aggregates(Xf)  # Add inter-station aggregates (mean, std, min, max, range)
    Xf = add_temp_delta_24h(Xf)      # Add 24h temperature delta per station (including Bern)
    return Xf

In [ ]:
# Build X2: original features + newly engineered features:
X2 = add_fe_features_basic(X)
y2 = y.copy()

# Keep the same train/test split as before:
X2_train = X2.loc[X_train.index]
X2_test = X2.loc[X_test.index]
y2_train = y_train
y2_test = y_test

# Identify engineered columns (columns added by FE only):
_engineered_cols_all = [c for c in X2.columns if c not in X.columns]
print("\nThe set of engineered features is:", len(_engineered_cols_all), "new columns.\n")
print(_engineered_cols_all[:10], "..." if len(_engineered_cols_all) > 10 else "")

---
## 5) Model training and diagnostic (no feature engineering exploitation)

### 5.1) Linear model

We start by building a linear model using only the main features to predict the targets. This provides a robust and interpretable solution for our prediction problem, serving primarily as a baseline and reference point for more complex models.

As the goal here is not to maximize predictive performance, we skipped a detailed linear correlation analysis between features and targets in the EDA. It would have unnecessarily complicated the report.

In our linear regression, using all 92 features could lead to overfitting. Therefore, we only include features that show a significant correlation with the targets. Additionally, we refine the feature set to avoid quasi-perfect multicollinearity, using the `reduced_features` variable with a correlation threshold of 0.95. This stabilizes the coefficients, simplifies their interpretation, and allows us to capture the true effects of the explanatory variables.

**Note:** `reduced_features` and `main_features_**h` introduce some slight data-leakage, as they were computed before the train/test split. However, this is not problematic, as the linear regression is only a baseline.

In [ ]:
print("\n" + "=" * 60)
print("MODEL TRAINING AND DIAGNOSTIC : LINEAR MODEL")
print("=" * 60)

# Cross-validation scheme:
cv_folds = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)  # we set fixed folds for comparability

reduced_features_model = reduce_multicollinearity(dataset, features, targets, 0.95)

Because the pipeline has not yet been applied to `X_train`, this variable does not contain the encoded season features.

For this reason, we normalize the variable `features` in order to ensure consistency between `feature_map` and `X_train`.

**Note:** Only `season` needs normalization in feature_map, as `hour_sin` and `hour_cos` are not among the selected features according to the EDA plots.

In [ ]:
def normalize_features(features):
    features = set(features)
    season_encoded = {f for f in features if f.startswith("season_")}
    if season_encoded:
        features = features - season_encoded
        features.add("season")
    return list(features)


feature_map_linear = {
    "target_air_temp_2m_plus12h": normalize_features(
        set(reduced_features_model).intersection(main_features_12h)
    ),
    "target_air_temp_2m_plus24h": normalize_features(
        set(reduced_features_model).intersection(main_features_24h)
    ),
    "target_air_temp_2m_plus48h": normalize_features(
        set(reduced_features_model).intersection(main_features_48h)
    )
}

In [ ]:
summary_linear = []
fig, axes = plt.subplots(1, len(targets), figsize=(20, 6))
axes = np.atleast_1d(axes)

pipelines_linear = {}

for idx, target in enumerate(targets):
    features_to_use = feature_map_linear[target]

    X_tr = X_train[features_to_use]
    X_te = X_test[features_to_use]
    y_tr = y_train[target]
    y_te = y_test[target]

    numeric_cols_to_use = [c for c in numeric_cols if c in features_to_use]
    preprocessor = make_preprocessor(features_to_use, numeric_cols_to_use)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    pipeline, mae_train_cv, mae_val_cv, mae_train_final, y_pred_te = evaluate_model(
        pipeline, X_tr, y_tr, X_te, cv_folds
    )

    pipelines_linear[target] = pipeline

    summary_linear.append({
        "Target": target,
        "MAE CV (train)": mae_train_cv,
        "MAE CV (validation)": mae_val_cv
    })

    model_coef = pipeline.named_steps["model"].coef_
    feature_names = get_feature_names_from_preprocessor(preprocessor, X_tr)

    plot_top_features(
        axes[idx],
        feature_names,
        model_coef,
        f"{target}\nIntercept={pipeline.named_steps['model'].intercept_:.2f} "
        f"| train MAE (refit)={mae_train_final:.2f}"
    )

plt.suptitle(
    "Linear Regression – Top-10 Feature Coefficients",
    fontsize=16,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    "linear_regression_pipeline.jpg",
    dpi=300,
    bbox_inches='tight'
)
plt.close()

print(
    "\nLinear regression plots saved under: "
    "linear_regression_pipeline.jpg\n"
)

summary_df = pd.DataFrame(summary_linear)
print("\nLinear Regression Summary :\n")
print(summary_df.to_string(index=False))

**Residual plots** (only used for model diagnostic, not comparison):

In [ ]:
def plot_residuals_pipeline_multi(
        pipelines,
        X_test,
        y_test,
        feature_map=None,
        model_name=""
):
    n_targets = len(pipelines)
    fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5))
    axes = np.atleast_1d(axes)

    for ax, (target, pipeline) in zip(axes, pipelines.items()):
        features_to_use = feature_map.get(target,
                                          X_test.columns.tolist()) if feature_map is not None else X_test.columns.tolist()
        X_te = X_test[features_to_use]
        y_te = y_test[target]

        y_pred = pipeline.predict(X_te)
        residuals = y_te - y_pred
        mae = mean_absolute_error(y_te, y_pred)

        sns.histplot(residuals, kde=True, bins=30, ax=ax)
        ax.axvline(0, color='red', linestyle='--')
        ax.set_title(f"{target}\nTest MAE = {mae:.2f}")
        ax.set_xlabel("Residuals")
        ax.grid(alpha=0.3)

    plt.suptitle(
        f"Residuals Distribution – {model_name}",
        fontsize=16,
        fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        f"residuals_{model_name.replace(' ', '_').lower()}.jpg",
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()

    print(
        f"\nDiagnostic plots for {model_name} have been saved under: "
        f"residuals_{model_name.replace(' ', '_').lower()}.jpg\n"
    )


plot_residuals_pipeline_multi(
    pipelines=pipelines_linear,
    X_test=X_test,
    y_test=y_test,
    feature_map=feature_map_linear,
    model_name="Linear Regression"
)

### 5.2) Ridge model

Given that we have high multicollinearity in our data, Ridge regression is better appropriated than Lasso.

Ridge penalisation will allow us to incorporate all features without overfitting (at the cost of a slight compromise on bias).

**Note:** from now on, for better prediction power, we will fit the model on all the features.

In [ ]:
print("\n" + "=" * 60)
print("MODEL TRAINING AND DIAGNOSTIC : RIDGE REGRESSION")
print("=" * 60)

summary_ridge = []
fig, axes = plt.subplots(1, len(targets), figsize=(20, 6))
axes = np.atleast_1d(axes)

ridge_grid = {"model__alpha": np.geomspace(1e-4, 1e6, 100)}
pipelines_ridge = {}

feature_map_ridge = {t: X_train.columns.tolist() for t in targets}

for idx, target in enumerate(targets):
    features_to_use = feature_map_ridge[target]

    X_tr = X_train[features_to_use]
    X_te = X_test[features_to_use]
    y_tr = y_train[target]
    y_te = y_test[target]

    numeric_cols_to_use = [c for c in numeric_cols if c in features_to_use]
    preprocessor = make_preprocessor(features_to_use, numeric_cols_to_use, scale_numeric=True)

    ridge_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", Ridge())
    ])

    grid = GridSearchCV(
        ridge_pipeline,
        param_grid=ridge_grid,
        scoring="neg_mean_absolute_error",
        cv=cv_folds,
        return_train_score=True,
        n_jobs=1
    )

    grid.fit(X_tr, y_tr)

    best_model_ridge = grid.best_estimator_
    best_alpha = grid.best_params_["model__alpha"]

    _, mae_train_cv, mae_val_cv, mae_train_final, y_pred_te = evaluate_model(
        best_model_ridge, X_tr, y_tr, X_te, cv_folds
    )

    pipelines_ridge[target] = best_model_ridge

    summary_ridge.append({
        "Target": target,
        "MAE CV (train)": mae_train_cv,
        "MAE CV (validation)": mae_val_cv,
        "Best alpha": best_alpha
    })

    ridge_preprocessor = best_model_ridge.named_steps["preprocessor"]
    feature_names = get_feature_names_from_preprocessor(ridge_preprocessor, X_tr)
    coef = best_model_ridge.named_steps["model"].coef_

    plot_top_features(
        axes[idx],
        feature_names,
        coef,
        f"{target}\nIntercept={best_model_ridge.named_steps['model'].intercept_:.2f} "
        f"| train MAE (refit)={mae_train_final:.2f}"
    )

plt.suptitle(
    "Ridge Regression – Top-10 Feature Coefficients",
    fontsize=16,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    "ridge_regression_pipeline.jpg",
    dpi=300,
    bbox_inches='tight'
)
plt.close()

print(
    "\nRidge regression plots have been saved under: "
    "ridge_regression_pipeline.jpg\n"
)

summary_ridge_df = pd.DataFrame(summary_ridge)
print("\nRidge Regression Summary:\n")
print(summary_ridge_df.to_string(index=False))

In [ ]:
# Diagnostic plots:
plot_residuals_pipeline_multi(
    pipelines=pipelines_ridge,
    X_test=X_test,
    y_test=y_test,
    feature_map=feature_map_ridge,
    model_name="Ridge Regression"
)

### 5.3) Random forest

Unlike KNN, Random Forest does not suffer from the curse of dimensionality.

This non-parametric method is well-suited to capture non-linear trends in the data. For the 12-hours target in particular, where the bias is already low, Random Forest can provide a model that is both robust and stable, while still flexible enough to capture complex interactions among features.

**Note:** Hyperparameter tuning is performed by maximizing the out-of-bag (OOB) performance rather than using cross-validation, and CV is only relevant for the model comparison.

In [ ]:
print("\n" + "=" * 60)
print("MODEL TRAINING AND DIAGNOSTIC : RANDOM FOREST")
print("=" * 60)

summary_rf = []
fig, axes = plt.subplots(1, len(targets), figsize=(20, 6))
axes = np.atleast_1d(axes)

max_features_grid = [3, 5, 7, "sqrt"]
min_samples_leaf_grid = [1, 3, 5, 7]

pipelines_rf = {}
feature_map_rf = {t: X_train.columns.tolist() for t in targets}

# Number of trees for tuning and final model
N_ESTIMATORS_TUNING = 500
N_ESTIMATORS_FINAL = 2000

for idx, target in enumerate(targets):

    features_to_use = feature_map_rf[target]

    X_tr = X_train[features_to_use]
    X_te = X_test[features_to_use]
    y_tr = y_train[target]
    y_te = y_test[target]

    best_oob_mae = np.inf
    best_oob_r2 = -np.inf
    best_max_features = None
    best_min_samples_leaf = None

    numeric_cols_to_use = [c for c in numeric_cols if c in features_to_use]

    for max_feat in max_features_grid:
        for min_leaf in min_samples_leaf_grid:

            preprocessor = make_preprocessor(features_to_use, numeric_cols_to_use, scale_numeric=False) # no need to scale in tree-based models

            rf_pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", RandomForestRegressor(
                    n_estimators=N_ESTIMATORS_TUNING,
                    max_features=max_feat,
                    min_samples_leaf=min_leaf,
                    oob_score=True,
                    bootstrap=True,
                    random_state=42,
                    n_jobs=1
                ))
            ])

            rf_pipeline.fit(X_tr, y_tr)
            rf_model = rf_pipeline.named_steps["model"]

            oob_pred = rf_model.oob_prediction_
            mask = ~np.isnan(oob_pred)
            if mask.sum() == 0:
                continue

            oob_mae = mean_absolute_error(np.asarray(y_tr)[mask], oob_pred[mask])
            oob_r2 = rf_model.oob_score_

            if oob_mae < best_oob_mae:
                best_oob_mae = oob_mae
                best_oob_r2 = oob_r2
                best_max_features = max_feat
                best_min_samples_leaf = min_leaf

    preprocessor = make_preprocessor(features_to_use, numeric_cols_to_use, scale_numeric=False)
    best_pipeline_rf = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=N_ESTIMATORS_FINAL,
            max_features=best_max_features,
            min_samples_leaf=best_min_samples_leaf,
            oob_score=True,
            bootstrap=True,
            random_state=42,
            n_jobs=1
        ))
    ])

    cv_results = cross_validate(
        best_pipeline_rf,
        X_tr,
        y_tr,
        cv=cv_folds,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=1
    )
    mae_train_cv = -cv_results["train_score"].mean()
    mae_val_cv = -cv_results["test_score"].mean()

    best_pipeline_rf.fit(X_tr, y_tr)
    y_pred_tr = best_pipeline_rf.predict(X_tr)
    mae_train_final = mean_absolute_error(y_tr, y_pred_tr)

    pipelines_rf[target] = best_pipeline_rf

    rf_model = best_pipeline_rf.named_steps["model"]

    summary_rf.append({
        "Target": target,
        "MAE CV (train)": mae_train_cv,
        "MAE CV (validation)": mae_val_cv,
        "Best OOB MAE": best_oob_mae,
        "Best OOB R2": best_oob_r2,
        "max_features": rf_model.max_features,
        "min_samples_leaf": rf_model.min_samples_leaf
    })

    feature_names = get_feature_names_from_preprocessor(
        best_pipeline_rf.named_steps["preprocessor"],
        X_tr
    )

    importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

    top_features = importances.head(10)

    plot_top_features(
        axes[idx],
        top_features.index,
        top_features.values,
        f"{target}\ntrain MAE (refit)={mae_train_final:.2f}"
    )

plt.suptitle(
    "Random Forest – Top-10 Feature Importances (OOB tuned)",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "random_forest_top10_importances.jpg",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print(
    "\nRandom Forest plots (top 10 importances) have been saved under: "
    "random_forest_top10_importances.jpg\n"
)

summary_rf_df = pd.DataFrame(summary_rf)
print("\nRandom Forest Summary:\n")
print(summary_rf_df.to_string(index=False))

In [ ]:
# Diagnostic plots:
plot_residuals_pipeline_multi(
    pipelines=pipelines_rf,
    X_test=X_test,
    y_test=y_test,
    feature_map=feature_map_rf,
    model_name="Random Forest Regression"
)

### 5.4) Gradient Boosting

We now consider an alternative tree-based approach: Gradient Boosting.

This method is well suited to capture non-linear patterns and complex predictive relationships.

Compared to Random Forests, Gradient Boosting offers a more structured form of flexibility through sequential tree construction, while remaining computationally efficient in our setting, as we apply early stop.

Model comparison is performed using the same external cross-validation scheme as in previous sections. This ensures a fair and consistent out-of-sample comparison across all models.

In [ ]:
print("\n" + "=" * 60)
print("MODEL TRAINING AND DIAGNOSTIC : GRADIENT BOOSTING")
print("=" * 60)

def run_gradient_boosting(X_train, X_test, y_train, y_test, feature_map, targets, cv_folds,
                          BMAX=3000, LEARNING_RATE=0.03, MAX_DEPTH=4,
                          MIN_SAMPLES_LEAF=50, SUBSAMPLE=0.7,
                          VAL_FRAC=0.15, N_NO_CHANGE=50, TOL=1e-4,
                          plot_title="Gradient Boosting", save_plot=None,
                          plot_residuals=True):
    summary_list = []
    pipelines_dict = {}
    feature_map_dict = {}

    fig, axes = plt.subplots(1, len(targets), figsize=(20, 6))
    axes = np.atleast_1d(axes)

    for idx, target in enumerate(targets):
        features_to_use = feature_map[target]
        numeric_cols_to_use = [c for c in X_train.select_dtypes(include=[np.number]).columns
                               if c in features_to_use]

        X_tr = X_train[features_to_use]
        X_te = X_test[features_to_use]
        y_tr = y_train[target]
        y_te = y_test[target]

        preprocessor = make_preprocessor(
            features_to_use,
            numeric_cols_to_use,
            scale_numeric=False
        )

        gb_pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", GradientBoostingRegressor(
                loss="absolute_error",
                n_estimators=BMAX,
                learning_rate=LEARNING_RATE,
                max_depth=MAX_DEPTH,
                min_samples_leaf=MIN_SAMPLES_LEAF,
                subsample=SUBSAMPLE,
                validation_fraction=VAL_FRAC,
                n_iter_no_change=N_NO_CHANGE,
                tol=TOL,
                random_state=42
            ))
        ])

        gb_pipeline.fit(X_tr, y_tr)
        B_star = gb_pipeline.named_steps["model"].estimators_.shape[0]

        gb_refit = clone(gb_pipeline)
        gb_refit.named_steps["model"].set_params(
            n_estimators=B_star,
            n_iter_no_change=None
        )

        gb_refit, mae_train_cv, mae_val_cv, mae_train_final, y_pred_te = evaluate_model(
            gb_refit, X_tr, y_tr, X_te, cv_folds
        )
        mae_test_holdout = mean_absolute_error(y_te, y_pred_te)

        pipelines_dict[target] = gb_refit
        feature_map_dict[target] = features_to_use

        feature_names = get_feature_names_from_preprocessor(
            gb_refit.named_steps["preprocessor"], X_tr
        )
        importances = pd.Series(
            gb_refit.named_steps["model"].feature_importances_,
            index=feature_names
        ).sort_values(ascending=False)
        top_features = importances.head(10)

        plot_top_features(
            axes[idx],
            top_features.index,
            top_features.values,
            f"{target}\nB*={B_star} | train MAE={mae_train_final:.2f}"
        )

        summary_list.append({
            "Target": target,
            "MAE CV (train)": mae_train_cv,
            "MAE CV (validation)": mae_val_cv,
            "MAE holdout (test)": mae_test_holdout,
            "B* (early stop)": B_star,
            "learning_rate": LEARNING_RATE,
            "max_depth": MAX_DEPTH,
            "min_samples_leaf": MIN_SAMPLES_LEAF,
            "subsample": SUBSAMPLE,
            "#features": len(features_to_use)
        })

    plt.suptitle(
        f"{plot_title} – Top 10 Feature Importances",
        fontsize=16,
        fontweight="bold"
    )
    plt.tight_layout()
    if save_plot:
        plt.savefig(save_plot, dpi=300, bbox_inches="tight")
    plt.close()

    if plot_residuals:
        plot_residuals_pipeline_multi(
            pipelines=pipelines_dict,
            X_test=X_test,
            y_test=y_test,
            feature_map=feature_map_dict,
            model_name=plot_title
        )

    return summary_list, pipelines_dict, feature_map_dict

In [ ]:
feature_map_gb = {t: X_train.columns.tolist() for t in targets}

summary_gb, pipelines_gb, feature_map_gb = run_gradient_boosting(
    X_train, X_test, y_train, y_test,
    feature_map=feature_map_gb,
    targets=targets,
    cv_folds=cv_folds,
    plot_title="Gradient Boosting",
    save_plot="gradient_boosting_earlystop_top10_importances.jpg"
)
print(
    "\nGradient boosting plots (top 10 importances) have been saved under: "
    "gradient_boosting_earlystop_importances.jpg\n"
)
summary_gb_df = pd.DataFrame(summary_gb)
print("\nGradient Boosting Summary:\n")
print(summary_gb_df.to_string(index=False))

Ex post, Gradient Boosting was identified as the best-performing model. Therefore, a more detailed summary was provided for this model.

---
## 6) Feature engineering exploitation

As we said in section 4, our feature engineered feature set is better suited for non-parametric models.

In addition to this, we see from the model summaries in section 5 that the gradient boosting, among all models, is the one that minimizes the validation MAE.

For these reasons, we will apply gradient boosting to the engineered features + to the old non station-related features (we created the `run_gradient_boosting` function ex post, in order to simplify the code).

In [ ]:
print("\n" + "="*60)
print("MODEL TRAINING AND DIAGNOSTIC : GRADIENT BOOSTING (feature engineered)")
print("="*60)

base_cols_no_suffix = [c for c in X.columns if _station_suffix(c) is None]
feature_map_gb_fe = {target: list(dict.fromkeys(base_cols_no_suffix + _engineered_cols_all)) for target in targets}

summary_gb_fe, pipelines_gb_fe, _ = run_gradient_boosting(
    X2_train, X2_test, y2_train, y2_test,
    feature_map=feature_map_gb_fe,
    targets=targets,
    cv_folds=cv_folds,
    plot_title="Gradient Boosting + FE",
    save_plot="gradient_boosting_earlystop_FE_top10_importances.jpg"
)
print(
    "\nGradient boosting plots accounting for FE (top 10 importances) have been saved under: "
    "gradient_boosting_earlystop_FE_top10_importances.jpg\n"
)
summary_gb_fe_df = pd.DataFrame(summary_gb_fe)
print("\nGradient Boosting Summary (FE):\n")
print(summary_gb_fe_df.to_string(index=False))

---
## 7) Comparative models summary

In [ ]:
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

fig, axes = plt.subplots(len(targets), 1, figsize=(8, 2 * len(targets)))
axes = np.atleast_1d(axes)

for ax, target in zip(axes, targets):
    # Linear Regression
    linear_row = summary_df[
        summary_df["Target"] == target
        ][["MAE CV (validation)"]].copy()
    linear_row.insert(0, "Model", "Linear Regression")

    # Ridge Regression
    ridge_row = summary_ridge_df[
        summary_ridge_df["Target"] == target
        ][["MAE CV (validation)"]].copy()
    ridge_row.insert(0, "Model", "Ridge Regression")

    # Random Forest
    rf_row = summary_rf_df[
        summary_rf_df["Target"] == target
        ][["MAE CV (validation)"]].copy()
    rf_row.insert(0, "Model", "Random Forest")

    # Gradient Boosting
    gb_row = summary_gb_df[
        summary_gb_df["Target"] == target
        ][["MAE CV (validation)"]].copy()
    gb_row.insert(0, "Model", "Gradient Boosting")

    # Gradient Boosting + FE
    gb_fe_row = summary_gb_fe_df[
        summary_gb_fe_df["Target"] == target
        ][["MAE CV (validation)"]].copy()
    gb_fe_row.insert(0, "Model", "Gradient Boosting + FE")

    combined_target = pd.concat(
        [linear_row, ridge_row, rf_row, gb_row, gb_fe_row],
        ignore_index=True
    )

    ax.axis('tight')
    ax.axis('off')
    table = ax.table(
        cellText=combined_target.values,
        colLabels=combined_target.columns,
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.auto_set_column_width(col=list(range(len(combined_target.columns))))
    ax.set_title(
        f"Target: {target}",
        fontsize=12,
        fontweight='bold',
        pad=10
    )

plt.tight_layout()
plt.savefig(
    "combined_summary_all_targets_with_FE.jpg",
    dpi=300,
    bbox_inches='tight'
)
plt.close()

print(
    "\nThe comparative summary has been saved under: "
    "combined_summary_all_targets_with_FE.jpg\n"
)